# Sentiment Analysis

In the absence of large self-labelled data sets such as product reviews or tweets, Old English doesn’t lend itself to typical machine-learning approaches to sentiment analysis. However, a lexicon-based approach cannot rely on such ready-made resources as VADER, because these libraries lack Old English sentiment lexicons. That leaves us with two options: manual lexicon creation and bootstrapping using a seed lexicon. Let’s try out a rough approach to the latter by way of experiment.

An obvious choice of seed terms is the cardinal sins (OE _heafodleahtras_) and virtues (_heafodmægenu_ or _fyrmest mægenu_, but usually simply referred to using the general term _mægenu_). ECHOE `049B.29b.txt` prints these two sets in a single sentence each:

> þæt is gitsung and gifernes galnes and weamodnes unrotnes asolcennes gilpgeornes and ofermodignes  
> […]  
> þæt is rumheortnes and syfernes clænnes and modþwærnes glædnes and anrædnes sibgeornes and eadmodnes

`394.25.txt` treats them in more detail, listing the diverse manifestations of each sin and a more detailed index of virtuous countermeasures. Let’s see what we can do with these as seed terms.

We’ll make use of last week’s word embedding technique to infer what other terms are semantically similar to our seed list. Start by recreating last week’s model of word embeddings in ECHOE, so we can look up similarity scores for our seed words:

In [1]:
from pathlib import Path
from gensim.models import Word2Vec

In [2]:
echoe_path = Path.cwd().parent / 'corpora' / 'echoe-bare'
corpus = []
for i in echoe_path.glob('*txt'):
    document = open(i).read().splitlines()
    for sentence in document:
        corpus.append(sentence)
sentences = [[token for token in document.split()] for document in corpus]

In [3]:
model = Word2Vec(sentences=sentences, min_count=1, vector_size=300, workers=2, window=3, epochs=30)

In [4]:
model.wv.most_similar('deofol')

[('deoful', 0.77921462059021),
 ('deofel', 0.5930476784706116),
 ('diofol', 0.5723055005073547),
 ('sacerd', 0.5484526753425598),
 ('þeodfeond', 0.5161810517311096),
 ('mann', 0.5082292556762695),
 ('antecrist', 0.502476155757904),
 ('mon', 0.48513883352279663),
 ('dysyg', 0.48309895396232605),
 ('cyningc', 0.46796488761901855)]

I have manually culled all positively and negatively connoted forms from `394.25.txt` and stored them in `misc/sentiment/sins.txt` and `misc/sentiment/virtues.txt`. Load these as follows:

In [5]:
_sins_path = Path.cwd().parent / 'misc' / 'sentiment' / 'sins.txt'
_virtues_path = Path.cwd().parent / 'misc' / 'sentiment' / 'virtues.txt'
sins = open(_sins_path).read().splitlines()
virtues = open(_virtues_path).read().splitlines()

We’ll use ECHOE word embeddings to extend these lists:

In [6]:
bad = []
for word in sins:
    bad.append(word)
    hits = [k for k,v in model.wv.most_similar(word, topn=200) if v > 0.93]
    for hit in hits:
        bad.append(hit)
bad = list(set(bad))

In [7]:
good = []
for word in virtues:
    good.append(word)
    hits = [k for k,v in model.wv.most_similar(word, topn=200) if v > 0.93]
    for hit in hits:
        good.append(hit)
good = list(set(good))

Now we have a list of mostly holy word entitled `good` and a list of mostly wicked words `bad`:

In [8]:
good_top = ', '.join(good[:20])
bad_top = ', '.join(bad[:20])
print('"Good" words include: {}.'.format(good_top))
print('')
print('"Bad" words include: {}.'.format(bad_top))

"Good" words include: wuldrigenne, haligra, ælmysdæda, geðeowade, sybbgelegera, englum, heofona, swicedomes, hadbrycas, regolbryce, lufe, rihtre, gescead, mægslihtas, gehealdsumnesse, rædinge, reliquiasocnum, læcedomlice, willindlice, mynstrum.

"Bad" words include: leahtor, ueneratione, wætes, geunrotsud, orwennesse, asolcennys, gewilnað, swylt, feollon, witu, heafodleahter, swygað, deofle, worulde, ðrælriht, unrotnesse, generwde, yfelnes, forhogunge, agotenes.


As explained by [Bessa](https://www.knime.com/blog/lexicon-based-sentiment-analysis), the most simplistic sentiment score is obtained by subtracting the count of negative words from the count of positive words in a document, then dividing the result by the sum of positive and negative word counts. This yields a score between `-1` and `1`, negative if a “sinful” sentiment value was inferred, positive for a “virtuous” value. So let’s write up a function to do this for us. If no matches are found among either our sins or our virtues, we’ll assign a score of `0`:

In [9]:
def sentScore(document):
    posCounter = 0
    negCounter = 0
    for word in document.split():
        if word in bad:
            negCounter += 1
        if word in good:
            posCounter += 1
    if (posCounter + negCounter) > 0:
        score = (posCounter - negCounter) / (posCounter + negCounter)
    else:
        score = 0
    return print(str(round(score, 2)) + ': \t' + document)

Now all we need is a corpus of sentences to be scored. We can obtain sets of pseudo-self-labelled sentences by reading all sentences containing occurrences of “deofol” (“devil”) and “eadig” (“blessed”) into lists:

In [10]:
devil = []
blessed = []
for doc in echoe_path.glob('*txt'):
    for line in open(doc).read().splitlines():
        if 'deofol' in line:
            devil.append(line.rstrip())
        if 'eadig' in line:
            blessed.append(line.rstrip())

At this point we can produce scores for all sentences in our lists. Let's confine ourselves to the first twenty of each:

In [11]:
print("=== Scoring the 'devil' set (expecting negative scores): ===")
for i in devil[:20]:
    sentScore(i)

=== Scoring the 'devil' set (expecting negative scores): ===
-1.0: 	þi læs þe hi unware wurðan aredode and þonne to rædlice þurh deofol beswicene
-1.0: 	and se deofol forgifð þærtogeanes dysig þæt he wisdomes ne gyme ne wislice ne lybbe
-1.0: 	ongean þam andgyte se deofol forgifð stuntnysse
-1.0: 	ongean þam wislican ræde se wiðerræda deofol sylð receleasnysse his underþeoddum
-1.0: 	ongean þæs modes strængðe se manfulla deofol forgyfð abroðennysse þæt se mann abreoðe on ælcere neode nahtlice æfre
-1.0: 	ongean þam ingehyde se hetola deofol sylð nytennysse nahtlicum mannum
-0.33: 	ongean godes ege se gramlica deofol sylð dyrstignysse mid dwæslicum gebærum receleasum mannum mid modes unstæþþignysse
0.5: 	þonne magon æghweþer ge us heofona rices eadignesse geearnian ge we megan gesæliglice befleon þa deorcan and þa dimman stowe helletintrego þe deofol an wunaþ mid his weagesiþum and mid þam awergdum saulum þa þanne noldan healdan þises gewrites bebod þe dryhten self awrat
-1.0: 	forhwan 

In [12]:
print("=== Scoring the 'blessed' set (expecting positive scores): ===")
for i in blessed[:20]:
    sentScore(i)

=== Scoring the 'blessed' set (expecting positive scores): ===
0: 	and ælc mann bið eadig þe hæfð þone wisdom
0.5: 	þonne magon æghweþer ge us heofona rices eadignesse geearnian ge we megan gesæliglice befleon þa deorcan and þa dimman stowe helletintrego þe deofol an wunaþ mid his weagesiþum and mid þam awergdum saulum þa þanne noldan healdan þises gewrites bebod þe dryhten self awrat
1.0: 	eal he þrowude for us þæt he wolde us generian fram hellewite and us gelæden in þa ecan eadignesse
1.0: 	for þissum eorðlicum wisum ic sylle þa heofonlican for þissum lænan life þæt unlæne for þissum hwilwendlicum þa ecan for þyssum ungecorenum þæt gecorene and for þyssum earman rice ic sylle þæt eadige
1.0: 	and þære eadigan ceastre weras gefeoð and wynsumiað on lisse and on blisse and on rice and on ecum gefean
0: 	þa arn se eadiga iohannes to eallum þam apostolum and wæs cweðende to him
0: 	and þa gesawon hie and ealle þa þe þær wæron þæt se eadiga michael genam and þa slog on þæs huses duru and 

1. What strikes you about these results?
2. Can you think of ways to improve the routine?

Two aspects that stand out in these results are (1) the large proportion of neutral values, and (2) the scarcity of values other than `1`, `0`, and `-1`. This is because *only* the words in our lexicon are counted, so the score is often one over one (e.g. one positive word minus no negative words, divided by their sum), and most sentences lack our registered words altogether. This is why the approach introduced in [ch. 20](http://web.stanford.edu/~jurafsky/slp3/20.pdf) of Jurafsky and Martin is more promising: it too starts from an embeddings model, but it stays there, creating a semantic axis (vector) between the two centroids, against which the vectors of all other terms, and thus sentences, may be measured by way of cosine similarity. We can do the same here:

In [13]:
import numpy as np
from numpy.linalg import norm
#from sklearn.preprocessing import minmax_scale
bad_array = np.array([model.wv.get_vector(i) for i in bad])
good_array = np.array([model.wv.get_vector(i) for i in good])
bad_centroid = bad_array.mean(axis=0)
good_centroid = good_array.mean(axis=0)
semantic_axis = np.subtract(good_centroid,bad_centroid)
#semantic_axis = minmax_scale(semantic_axis.T).T 

In [14]:
from cltk.stops.ang import STOPS
def sentScore2(doc):
    stopped = [token for token in doc.split() if not token in STOPS]
    #doc_embedding = np.mean(np.array([model.wv.get_vector(i) for i in doc.split()]),axis=0)
    doc_embedding = np.mean(np.array([model.wv.get_vector(i) for i in stopped]),axis=0)
    cos_sim = semantic_axis.dot(doc_embedding) / (norm(semantic_axis) * norm(doc_embedding))
    return print(str(round(cos_sim, 2)) + ':\t ' + doc)

In [15]:
print("=== Scoring the 'devil' set (expecting lower scores): ===")
for i in devil[:20]:
    sentScore2(i)

=== Scoring the 'devil' set (expecting lower scores): ===
0.01:	 þi læs þe hi unware wurðan aredode and þonne to rædlice þurh deofol beswicene
0.03:	 and se deofol forgifð þærtogeanes dysig þæt he wisdomes ne gyme ne wislice ne lybbe
-0.0:	 ongean þam andgyte se deofol forgifð stuntnysse
0.03:	 ongean þam wislican ræde se wiðerræda deofol sylð receleasnysse his underþeoddum
0.04:	 ongean þæs modes strængðe se manfulla deofol forgyfð abroðennysse þæt se mann abreoðe on ælcere neode nahtlice æfre
0.06:	 ongean þam ingehyde se hetola deofol sylð nytennysse nahtlicum mannum
0.27:	 ongean godes ege se gramlica deofol sylð dyrstignysse mid dwæslicum gebærum receleasum mannum mid modes unstæþþignysse
0.46:	 þonne magon æghweþer ge us heofona rices eadignesse geearnian ge we megan gesæliglice befleon þa deorcan and þa dimman stowe helletintrego þe deofol an wunaþ mid his weagesiþum and mid þam awergdum saulum þa þanne noldan healdan þises gewrites bebod þe dryhten self awrat
0.09:	 forhwan ear

In [16]:
print("=== Scoring the 'blessed' set (expecting higher scores): ===")
for i in blessed[:20]:
    sentScore2(i)

=== Scoring the 'blessed' set (expecting higher scores): ===
0.13:	 and ælc mann bið eadig þe hæfð þone wisdom
0.46:	 þonne magon æghweþer ge us heofona rices eadignesse geearnian ge we megan gesæliglice befleon þa deorcan and þa dimman stowe helletintrego þe deofol an wunaþ mid his weagesiþum and mid þam awergdum saulum þa þanne noldan healdan þises gewrites bebod þe dryhten self awrat
0.33:	 eal he þrowude for us þæt he wolde us generian fram hellewite and us gelæden in þa ecan eadignesse
0.31:	 for þissum eorðlicum wisum ic sylle þa heofonlican for þissum lænan life þæt unlæne for þissum hwilwendlicum þa ecan for þyssum ungecorenum þæt gecorene and for þyssum earman rice ic sylle þæt eadige
0.4:	 and þære eadigan ceastre weras gefeoð and wynsumiað on lisse and on blisse and on rice and on ecum gefean
0.34:	 þa arn se eadiga iohannes to eallum þam apostolum and wæs cweðende to him
0.26:	 and þa gesawon hie and ealle þa þe þær wæron þæt se eadiga michael genam and þa slog on þæs huses

In [17]:
sentScore2('god halig sanctus')

0.31:	 god halig sanctus


In [18]:
sentScore('god halig sanctus')

1.0: 	god halig sanctus


In [19]:
sentScore2('yfel deofol')

-0.08:	 yfel deofol


In [20]:
sentScore('yfel deofol')

-1.0: 	yfel deofol


In [21]:
sentScore2('a sy lof and wuldor fæder and suna and halgum gaste')

0.6:	 a sy lof and wuldor fæder and suna and halgum gaste


In [22]:
sentScore('a sy lof and wuldor fæder and suna and halgum gaste')

1.0: 	a sy lof and wuldor fæder and suna and halgum gaste


In [23]:
sentScore2('lof wuldor fæder suna halgum gaste')

0.63:	 lof wuldor fæder suna halgum gaste


In [24]:
sentScore2('þæt is gitsung and gifernes galnes and weamodnes unrotnes asolcennes gilpgeornes and ofermodignes')

-0.18:	 þæt is gitsung and gifernes galnes and weamodnes unrotnes asolcennes gilpgeornes and ofermodignes


In [25]:
sentScore('þæt is gitsung and gifernes galnes and weamodnes unrotnes asolcennes gilpgeornes and ofermodignes')

-1.0: 	þæt is gitsung and gifernes galnes and weamodnes unrotnes asolcennes gilpgeornes and ofermodignes


In [26]:
sentScore2('gitsung gifernes galnes weamodnes unrotnes asolcennes gilpgeornes ofermodignes')

-0.18:	 gitsung gifernes galnes weamodnes unrotnes asolcennes gilpgeornes ofermodignes


To classify sentences between these poles, we'd have to decide on a threshold value. But our "pseudolabelled" sentences don't cluster as neatly as we might have hoped, because each word contributes equally to the sentence's sentiment value, and most of these sentences contain just a few words with strong polarity. Even just (manually) removing the stop words in that last test made a significant difference. Thus excluding stop words seems useful when calculating sentiment, though it would be better still to identify common _n_-grams before we do so.